In [8]:
import pandas as pd 
import numpy as np
from s3_utils import read_s3_csv, upload_to_s3, get_s3_client
import re

In [9]:
s3 = get_s3_client()
BUCKET = "projet-accidents-jedha"

# 1. On liste TOUS les fichiers du dossier bronze
response = s3.list_objects_v2(Bucket=BUCKET, Prefix='bronze/')

# 2. On crée une liste vide pour stocker nos DataFrames
dfs_to_merge = []

# 3. La boucle magique intelligente
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    # On filtre pour ne garder que la catégorie "caract"
    if "caract" in file_key.lower():
        
        # --- LOGIQUE D'EXTRACTION DE L'ANNÉE ---
        # On cherche 4 chiffres dans le nom (ex: 2024)
        match = re.search(r'(\d{4})', file_key)
        
        if match:
            annee = int(match.group(1))
            
            # On définit le séparateur selon l'année
            if annee >= 2024:
                sep = ','
            else:
                sep = ';'
            
            print(f"🔎 Trouvé : {file_key} | Année: {annee} | Séparateur: '{sep}'")
            
            # On lit le fichier avec le bon séparateur
            df_temp = read_s3_csv(file_key, separator=sep)
            dfs_to_merge.append(df_temp)

# 4. On fusionne tout d'un coup
if dfs_to_merge:
    caract_df = pd.concat(dfs_to_merge, ignore_index=True)
    print(f"✅ Terminé ! {len(dfs_to_merge)} fichiers fusionnés.")
else:
    print("⚠️ Aucun fichier trouvé, vérifie le mot-clé ou le dossier.")

🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - caract 2021.csv | Année: 2021 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - caract 2022.csv | Année: 2022 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - caract 2023.csv | Année: 2023 | Séparateur: ';'
🔎 Trouvé : bronze/Dataset - Sécurité Routière.xlsx - caract 2024.csv | Année: 2024 | Séparateur: ','
✅ Terminé ! 4 fichiers fusionnés.


In [10]:
caract_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221044 entries, 0 to 221043
Data columns (total 16 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Num_Acc      165742 non-null  float64
 1   jour         221044 non-null  int64  
 2   mois         221044 non-null  int64  
 3   an           221044 non-null  int64  
 4   hrmn         221044 non-null  object 
 5   lum          221044 non-null  int64  
 6   dep          221044 non-null  object 
 7   com          221044 non-null  object 
 8   agg          221044 non-null  int64  
 9   int          221044 non-null  int64  
 10  atm          221044 non-null  int64  
 11  col          221044 non-null  int64  
 12  adr          215539 non-null  object 
 13  lat          221044 non-null  object 
 14  long         221044 non-null  object 
 15  Accident_Id  55302 non-null   float64
dtypes: float64(2), int64(8), object(6)
memory usage: 27.0+ MB


In [11]:
# 1. On répare l'heure (hrmn) pour tout le monde d'un coup
h_fix = caract_df['hrmn'].astype(str).str.replace(':', '').str.replace('nan', '0000').str.zfill(4)

# 2. On crée la date en utilisant caract_df (au lieu de df_2024)
# Comme ça, on traite 2021, 2022, 2023 ET 2024 en une seule fois !
date_string = (
    caract_df['an'].astype(str) + '-' + 
    caract_df['mois'].astype(str) + '-' + 
    caract_df['jour'].astype(str) + ' ' + 
    h_fix.str[:2] + ':' + h_fix.str[2:]
)

# 3. On transforme en vrai format date
caract_df['date'] = pd.to_datetime(date_string, errors='coerce')

# 4. On check les erreurs
errors = caract_df['date'].isna().sum()
print(f"Nombre de dates invalides : {errors}")

# 5. On nettoie les colonnes devenues inutiles
caract_df = caract_df.drop(columns=['an', 'mois', 'jour', 'hrmn', 'adr'])

print(caract_df.head())

Nombre de dates invalides : 0
        Num_Acc  lum dep    com  agg  int  atm  col            lat  \
0  2.021000e+11    2  30  30319    1    1    1    1  44,0389580000   
1  2.021000e+11    1  51  51544    1    3    1    3  49,2421290000   
2  2.021000e+11    1  85  85048    2    1    7    6  46,9219500000   
3  2.021000e+11    5  93  93005    2    2    3    6  48,9493634583   
4  2.021000e+11    5  76  76429    2    1    1    2  49,4083800000   

             long  Accident_Id                date  
0    4,3480220000          NaN 2021-11-30 07:32:00  
1    4,5545460000          NaN 2021-09-25 14:20:00  
2   -0,9644600000          NaN 2021-07-15 07:55:00  
3    2,5196639908          NaN 2021-03-27 19:45:00  
4    1,1458100000          NaN 2021-02-25 07:20:00  


In [12]:
caract_df

,Num_Acc,lum,dep,com,agg,int,atm,col,lat,long,Accident_Id,date
0,2.021000e+11,2,30,30319,1,1,1,1,"44,0389580000","4,3480220000",NaN,2021-11-30 07:32:00
1,2.021000e+11,1,51,51544,1,3,1,3,"49,2421290000","4,5545460000",NaN,2021-09-25 14:20:00
2,2.021000e+11,1,85,85048,2,1,7,6,"46,9219500000","-0,9644600000",NaN,2021-07-15 07:55:00
3,2.021000e+11,5,93,93005,2,2,3,6,"48,9493634583","2,5196639908",NaN,2021-03-27 19:45:00
4,2.021000e+11,5,76,76429,2,1,1,2,"49,4083800000","1,1458100000",NaN,2021-02-25 07:20:00
...,...,...,...,...,...,...,...,...,...,...,...,...
221039,2.024001e+11,5,94,94065,2,3,2,6,48.7574,2.34469,NaN,2024-07-10 02:05:00
221040,2.024001e+11,1,92,92062,2,1,1,6,48.886943,2.248636,NaN,2024-11-30 15:27:00
221041,2.024001e+11,3,29,29068,1,1,1,1,48.52619,-3.963176,NaN,2024-10-24 20:40:00
221042,2.024001e+11,1,92,92012,2,1,1,3,48.832862,2.24394,NaN,2024-11-30 15:40:00


In [13]:
# Je garde ce bout de code au cas ou car je ne sais pas si j'ai le droit de le suprimmé. je le mets en stase.

"""h_fix = caract_df['hrmn'].astype(str).str.replace(':', '').str.replace('nan', '0000').str.zfill(4)

date_string = (
    df_2024['an'].astype(str) + '-' + 
    df_2024['mois'].astype(str) + '-' + 
    df_2024['jour'].astype(str) + ' ' + 
    h_fix.str[:2] + ':' + h_fix.str[2:]
)

caract_df['date'] = pd.to_datetime(date_string, errors='coerce')

errors = caract_df['date'].isna().sum()
print(f"Nombre de dates invalides : {errors}")

caract_df = caract_df.drop(columns=['an', 'mois', 'jour', 'hrmn', 'adr'])

print(caract_df.head())"""

'h_fix = caract_df[\'hrmn\'].astype(str).str.replace(\':\', \'\').str.replace(\'nan\', \'0000\').str.zfill(4)\n\ndate_string = (\n    df_2024[\'an\'].astype(str) + \'-\' + \n    df_2024[\'mois\'].astype(str) + \'-\' + \n    df_2024[\'jour\'].astype(str) + \' \' + \n    h_fix.str[:2] + \':\' + h_fix.str[2:]\n)\n\ncaract_df[\'date\'] = pd.to_datetime(date_string, errors=\'coerce\')\n\nerrors = caract_df[\'date\'].isna().sum()\nprint(f"Nombre de dates invalides : {errors}")\n\ncaract_df = caract_df.drop(columns=[\'an\', \'mois\', \'jour\', \'hrmn\', \'adr\'])\n\nprint(caract_df.head())'

In [14]:
caract_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221044 entries, 0 to 221043
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Num_Acc      165742 non-null  float64       
 1   lum          221044 non-null  int64         
 2   dep          221044 non-null  object        
 3   com          221044 non-null  object        
 4   agg          221044 non-null  int64         
 5   int          221044 non-null  int64         
 6   atm          221044 non-null  int64         
 7   col          221044 non-null  int64         
 8   lat          221044 non-null  object        
 9   long         221044 non-null  object        
 10  Accident_Id  55302 non-null   float64       
 11  date         221044 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(5), object(4)
memory usage: 20.2+ MB


In [15]:
caract_df = caract_df.drop(columns=['Accident_Id'])

In [16]:
# code ne fonctionne pas. je l'ai mis en stase.

"""caract_df = caract_df.drop(columns=['an', 'mois', 'jour', 'adr'])
print(caract_df.info())"""

"caract_df = caract_df.drop(columns=['an', 'mois', 'jour', 'adr'])\nprint(caract_df.info())"

In [17]:
caract_df

,Num_Acc,lum,dep,com,agg,int,atm,col,lat,long,date
0,2.021000e+11,2,30,30319,1,1,1,1,"44,0389580000","4,3480220000",2021-11-30 07:32:00
1,2.021000e+11,1,51,51544,1,3,1,3,"49,2421290000","4,5545460000",2021-09-25 14:20:00
2,2.021000e+11,1,85,85048,2,1,7,6,"46,9219500000","-0,9644600000",2021-07-15 07:55:00
3,2.021000e+11,5,93,93005,2,2,3,6,"48,9493634583","2,5196639908",2021-03-27 19:45:00
4,2.021000e+11,5,76,76429,2,1,1,2,"49,4083800000","1,1458100000",2021-02-25 07:20:00
...,...,...,...,...,...,...,...,...,...,...,...
221039,2.024001e+11,5,94,94065,2,3,2,6,48.7574,2.34469,2024-07-10 02:05:00
221040,2.024001e+11,1,92,92062,2,1,1,6,48.886943,2.248636,2024-11-30 15:27:00
221041,2.024001e+11,3,29,29068,1,1,1,1,48.52619,-3.963176,2024-10-24 20:40:00
221042,2.024001e+11,1,92,92012,2,1,1,3,48.832862,2.24394,2024-11-30 15:40:00


In [18]:
nouveaux_noms = {
    'Num_ACC': 'luminosite',
    'dep': 'departement',
    'com': 'commune',
    'agg': 'agglomeration',
    'int': 'intersection',
    'atm': 'meteo',
    'col': 'type_collision'
}

# On applique le changement
caract_df = caract_df.rename(columns=nouveaux_noms)

# On vérifie le résultat
print(caract_df.columns)
caract_df.head()

Index(['Num_Acc', 'lum', 'departement', 'commune', 'agglomeration',
       'intersection', 'meteo', 'type_collision', 'lat', 'long', 'date'],
      dtype='object')


,Num_Acc,lum,departement,commune,agglomeration,intersection,meteo,type_collision,lat,long,date
0,2.021000e+11,2,30,30319,1,1,1,1,"44,0389580000","4,3480220000",2021-11-30 07:32:00
1,2.021000e+11,1,51,51544,1,3,1,3,"49,2421290000","4,5545460000",2021-09-25 14:20:00
2,2.021000e+11,1,85,85048,2,1,7,6,"46,9219500000","-0,9644600000",2021-07-15 07:55:00
3,2.021000e+11,5,93,93005,2,2,3,6,"48,9493634583","2,5196639908",2021-03-27 19:45:00
4,2.021000e+11,5,76,76429,2,1,1,2,"49,4083800000","1,1458100000",2021-02-25 07:20:00


In [19]:
colonnes_a_traiter = ['meteo', 'type_collision']
caract_df[colonnes_a_traiter] = caract_df[colonnes_a_traiter].replace(-1, np.nan)
caract_df

,Num_Acc,lum,departement,commune,agglomeration,intersection,meteo,type_collision,lat,long,date
0,2.021000e+11,2,30,30319,1,1,1.0,1.0,"44,0389580000","4,3480220000",2021-11-30 07:32:00
1,2.021000e+11,1,51,51544,1,3,1.0,3.0,"49,2421290000","4,5545460000",2021-09-25 14:20:00
2,2.021000e+11,1,85,85048,2,1,7.0,6.0,"46,9219500000","-0,9644600000",2021-07-15 07:55:00
3,2.021000e+11,5,93,93005,2,2,3.0,6.0,"48,9493634583","2,5196639908",2021-03-27 19:45:00
4,2.021000e+11,5,76,76429,2,1,1.0,2.0,"49,4083800000","1,1458100000",2021-02-25 07:20:00
...,...,...,...,...,...,...,...,...,...,...,...
221039,2.024001e+11,5,94,94065,2,3,2.0,6.0,48.7574,2.34469,2024-07-10 02:05:00
221040,2.024001e+11,1,92,92062,2,1,1.0,6.0,48.886943,2.248636,2024-11-30 15:27:00
221041,2.024001e+11,3,29,29068,1,1,1.0,1.0,48.52619,-3.963176,2024-10-24 20:40:00
221042,2.024001e+11,1,92,92012,2,1,1.0,3.0,48.832862,2.24394,2024-11-30 15:40:00


In [ ]:
# caract_df.to_csv('caract 2021-2024.csv', index=False, sep=';', encoding='utf-8')

# On utilise ta fonction d'upload
# 'caract_clean.csv' est le nom que le fichier aura sur S3
upload_to_s3(caract_df, "caracteristiques_cleaned.csv", folder="silver")

✅ Mission accomplie : silver/caracteristiques_cleaned_test.csv est sur S3 !


In [21]:
###coucou Paul
